## Step 0. 환경 세팅 및 데이터 준비
`bcftools`를 설치하고, 환자 VCF와 **GRCh37(hg19) 버전 ClinVar**를 내려받습니다.

In [ ]:
# 1. VCF 분석 도구 설치
!apt-get update -qq && apt-get install -y bcftools tabix

# 2. 환자 VCF 다운로드
!wget -qO patient.vcf.gz https://github.com/swjung-bi/vcf-sc-practice/raw/refs/heads/main/patient.vcf.gz
!bcftools index -t patient.vcf.gz

# 3. ClinVar (GRCh37 = hg19) 다운로드 및 인덱싱
!wget -qO clinvar.vcf.gz "https://ftp.ncbi.nlm.nih.gov/pub/clinvar/vcf_GRCh37/clinvar.vcf.gz"
!tabix -p vcf clinvar.vcf.gz
print("파일 다운로드 및 인덱싱 완료")

## Step 1. 환자 VCF 탐색
환자의 변이 목록을 살펴봅니다. 각 줄은 하나의 변이이며,
염색체(CHROM)·위치(POS)·정상 염기(REF)·변이 염기(ALT) 정보를 담고 있습니다.

In [ ]:
import pandas as pd, io, subprocess
from IPython.display import display

# VCF를 표(DataFrame)로 읽기
# (## 메타데이터 줄은 제외하고, #CHROM 줄을 컬럼명으로 사용)
txt = subprocess.run(["bcftools", "view", "patient.vcf.gz"],
                     capture_output=True, text=True).stdout
rows = [l for l in txt.split("\n") if l and not l.startswith("##")]
vcf = pd.read_csv(io.StringIO("\n".join(rows)), sep="\t")
vcf = vcf.rename(columns={"#CHROM": "CHROM"})

print(f"환자의 총 변이 수: {len(vcf):,}")
print("칼럼 구성:", list(vcf.columns))
print("\n앞 10개 변이:")
display(vcf.head(10))

## Step 2. 환자 VCF 필터링
시퀀싱 오류로 기록된 저품질 변이를 먼저 걷어냅니다.


In [ ]:
# 하드 필터링: 저품질 변이 제거 (QUAL ≤ 30 탈락)
!bcftools view -i 'QUAL>30' patient.vcf.gz -Oz -o patient.filtered.vcf.gz
!bcftools index -t patient.filtered.vcf.gz
print("필터 전후 변이 수:")
!echo -n "  전: "; bcftools view -H patient.vcf.gz | wc -l
!echo -n "  후: "; bcftools view -H patient.filtered.vcf.gz | wc -l

## Step 3. 임상 데이터베이스 주석 (ClinVar)
날것의 A/G/C/T 변이만으로는 질병 유발 여부를 알 수 없습니다. 전 세계 임상 데이터가 모인
**ClinVar**를 대조해, 각 변이에 병원성(CLNSIG)·질병명(CLNDN)을 입힙니다.
환자 VCF와 ClinVar 모두 접두사 없는 염색체명(`17`)을 써서 그대로 매칭됩니다.

In [ ]:
# ClinVar에서 CLNSIG·CLNDN을 환자 VCF에 복사
!bcftools annotate -a clinvar.vcf.gz -c INFO/CLNSIG,INFO/CLNDN patient.filtered.vcf.gz -Oz -o patient.annotated.vcf.gz
!bcftools index -t patient.annotated.vcf.gz
print("주석 완료")

## Step 4. 병원성 변이 필터링
주석 결과의 분포를 먼저 보고(대부분은 양성!), **명확히 'Pathogenic/Likely_pathogenic'**
으로 보고된 변이만 추립니다. (`Conflicting...`(상충/불명)은 제외)

In [ ]:
import pandas as pd, io, subprocess

# 주석된 VCF를 표(DataFrame)로 읽기
txt = subprocess.run(["bcftools", "view", "patient.annotated.vcf.gz"],
                     capture_output=True, text=True).stdout
rows = [l for l in txt.split("\n") if l and not l.startswith("##")]
df = pd.read_csv(io.StringIO("\n".join(rows)), sep="\t")

# INFO 칸에서 CLNSIG/CLNDN 값 꺼내기
def info_get(info, key):
    for field in str(info).split(";"):
        if field.startswith(key + "="):
            return field.split("=", 1)[1]
    return None

df["CLNSIG"] = df["INFO"].apply(lambda x: info_get(x, "CLNSIG"))
df["CLNDN"]  = df["INFO"].apply(lambda x: info_get(x, "CLNDN"))

clin = df[df["CLNSIG"].notna()].copy()
print(f"전체 변이 {len(df):,}개 중 ClinVar 등록 변이 {len(clin):,}개\n")
print("ClinVar 분류 분포 (상위 8개):")
print(clin["CLNSIG"].value_counts().head(8).to_string())

def first_disease(clndn):
    skip = {"not_specified", "not_provided", "."}
    for d in str(clndn).split("|"):
        if d not in skip:
            return d
    return "(질병명 미지정)"
patho["DISEASE"] = patho["CLNDN"].apply(first_disease)

# 명확한 병원성만 (Conflicting 등은 제외)
patho = clin[clin["CLNSIG"].str.match(r"(Pathogenic|Likely_pathogenic)", na=False)].copy()
patho["DISEASE"] = patho["CLNDN"].str.split("|").str[0]   # 대표 질병명 한 개만
print(f"\n 명확한 병원성 변이 {len(patho)}개:")
print(patho[["#CHROM", "POS", "REF", "ALT", "CLNSIG", "DISEASE"]].to_string(index=False))

## Step 5. 임상 표현형으로 원인 변이 선택 및 해석
병원성 변이가 둘 이상 나올 수 있습니다(예: 우연히 발견된 다른 질환의 변이).
환자의 임상 표현형(**유방암**)에 맞는 변이를 골라 진단을 확정합니다.

In [ ]:
# CLNDN에 BRCA1 / Breast 가 포함된 병원성 변이 = 이 환자의 원인 변이
cause = patho[patho["CLNDN"].str.contains("BRCA1|Breast", case=False, na=False)]

print(" 임상 표현형(유방암)과 일치하는 원인 변이:")
print(cause[["#CHROM", "POS", "REF", "ALT", "CLNSIG"]].to_string(index=False))

print("\n관련 질병명(CLNDN):")
for d in cause.iloc[0]["CLNDN"].split("|")[:6]:
    print("  -", d)